# Moments of a Distribution

The expected value and the variance paint a really good picture of a distribution — but they miss subtleties like lopsidedness (**skewness**) and tail heaviness (**kurtosis**). The gateway to those finer descriptors is the family of **moments**: $E[X]$, $E[X^2]$, $E[X^3]$, … , $E[X^k]$. This short notebook formalizes them, shows how the first two recover the mean and variance, and previews why the higher ones matter. It is based on the ninth video of Lesson 3 (Describing Distributions).

## 1. Introduction

Two distributions can share the same mean *and* the same variance and still look very different — one leaning left, one leaning right; one with slim tails, one prone to extreme values. To tell them apart you need numbers beyond $E[X]$ and $\mathrm{Var}(X)$.

The pattern is hiding in what you have already computed. $E[X]$ and $E[X^2]$ appeared throughout this lesson (the shortcut variance formula is built from exactly these two). Why stop at the square?

After completing this notebook you will be able to:

- Define the **$k$-th moment** $E[X^k]$ and compute it for discrete distributions.
- Relate the first two moments to the mean and variance.
- Distinguish **raw** moments from **central** moments (moments of the centered variable).
- Explain, in preview, which moments feed skewness and kurtosis.

## 2. Theory

### A worked warm-up

Take a random variable with

| $x$ | $-2$ | $0$ | $1$ |
|---|---|---|---|
| $p(x)$ | $\tfrac13$ | $\tfrac16$ | $\tfrac12$ |

Its expectation is the familiar weighted average:

$$
E[X] = \tfrac13(-2) + \tfrac16(0) + \tfrac12(1) = -\tfrac16
$$

And the expectation of the variable *squared* — same recipe, squared values, same probabilities:

$$
E[X^2] = \tfrac13(-2)^2 + \tfrac16(0)^2 + \tfrac12(1)^2 = \tfrac43 + \tfrac12 = \tfrac{11}{6}
$$

### The moments

These two quantities have names:

- $E[X]$ is the **first moment** — the expectation.
- $E[X^2]$ is the **second moment** — *related* to the variance (not exactly the variance: you have to center it first, since $\mathrm{Var}(X) = E[X^2] - E[X]^2$).

What goes next? Keep raising the power:

- $E[X^3]$ is the **third moment**,
- and in general the **$k$-th moment** is $E[X^k]$.

For a discrete variable taking values $x_1, \dots, x_n$ with probabilities $p_1, \dots, p_n$:

$$
\text{1st moment} = \sum_i p_i x_i, \qquad
\text{2nd moment} = \sum_i p_i x_i^2, \qquad
\text{3rd moment} = \sum_i p_i x_i^3, \qquad
\text{$k$-th moment} = \sum_i p_i x_i^k
$$

(For continuous variables the sums become integrals against the PDF, as always.)

### Raw vs. central moments

The moments above are **raw** moments — powers of $X$ itself. As the video hints for the variance, the more descriptive quantities usually come from *centering first*: the **$k$-th central moment** is

$$
E\big[(X - \mu)^k\big]
$$

- 1st central moment: $E[X - \mu] = 0$ — always (that was the "average deviation" dead end).
- 2nd central moment: $E[(X-\mu)^2]$ — exactly the **variance**.
- 3rd central moment: measures *asymmetry* — the ingredient of **skewness**.
- 4th central moment: measures *tail weight* — the ingredient of **kurtosis**.

That is why the moments "are going to be pretty useful in a bit": the next videos build skewness and kurtosis out of the third and fourth moments.

## 3. Worked Examples

### Example 1: all four moments of the warm-up variable

$$
E[X] = -\tfrac16, \qquad
E[X^2] = \tfrac{11}{6}, \qquad
E[X^3] = \tfrac13(-8) + \tfrac12(1) = -\tfrac{13}{6}, \qquad
E[X^4] = \tfrac13(16) + \tfrac12(1) = \tfrac{35}{6}
$$

Even powers are dominated by *large magnitudes* (the $-2$ contributes $+16$ to the fourth moment); odd powers keep the *sign* (the $-2$ drags the third moment negative). This sign-keeping is exactly what will make odd moments detect asymmetry.

### Example 2: mean and variance from the first two moments

$$
\mathrm{Var}(X) = E[X^2] - E[X]^2 = \tfrac{11}{6} - \left(-\tfrac16\right)^2 = \tfrac{66}{36} - \tfrac{1}{36} = \tfrac{65}{36} \approx 1.806
$$

The first two moments carry the same information as (mean, variance).

### Example 3: moments of a fair die

$$
E[X] = 3.5, \qquad E[X^2] = \tfrac{91}{6} \approx 15.17, \qquad E[X^3] = \tfrac{441}{6} = 73.5, \qquad E[X^4] = \tfrac{2275}{6} \approx 379.17
$$

(The first two are old friends from the earlier notebooks.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### A moment function — one recipe for every $k$

In [ ]:
def moment(values, pmf, k):
    return np.sum(pmf * values**k)

def central_moment(values, pmf, k):
    mu = moment(values, pmf, 1)
    return np.sum(pmf * (values - mu)**k)

# The video's example: -2 w.p. 1/3, 0 w.p. 1/6, 1 w.p. 1/2
values = np.array([-2, 0, 1])
pmf = np.array([1/3, 1/6, 1/2])

for k in range(1, 5):
    print(f"{k}-th moment E[X^{k}] = {moment(values, pmf, k):>8.4f}"
          f"   central E[(X-mu)^{k}] = {central_moment(values, pmf, k):>8.4f}")

print(f"\nVariance via moments: E[X^2] - E[X]^2 = "
      f"{moment(values, pmf, 2) - moment(values, pmf, 1)**2:.4f}  (= 65/36 = {65/36:.4f})")

### Moments of the fair die, exact and simulated

In [ ]:
rng = np.random.default_rng(42)

faces = np.arange(1, 7)
pmf = np.full(6, 1 / 6)
rolls = rng.integers(1, 7, size=1_000_000)

print(f"{'k':>3} {'exact E[X^k]':>13} {'simulated':>11}")
for k in range(1, 5):
    print(f"{k:>3} {moment(faces, pmf, k):>13.4f} {(rolls.astype(float)**k).mean():>11.4f}")

### Same mean and variance, different higher moments

Two distributions engineered to agree on the first two moments — the third moment is what tells them apart. (This is the preview of skewness.)

In [ ]:
# Distribution A: symmetric around 0. Distribution B: skewed, same mean & variance.
values_a = np.array([-1.0, 0.0, 1.0])
pmf_a = np.array([0.25, 0.5, 0.25])          # mean 0, variance 0.5

values_b = np.array([-1 / (2 * np.sqrt(2)), np.sqrt(2)])   # chosen so mean = 0, var = 0.5
pmf_b = np.array([0.8, 0.2])

for name, v, p in [("A (symmetric)", values_a, pmf_a), ("B (lopsided) ", values_b, pmf_b)]:
    m1 = moment(v, p, 1)
    var = central_moment(v, p, 2)
    m3c = central_moment(v, p, 3)
    print(f"{name}: mean = {m1:>7.4f}, variance = {var:.4f}, 3rd central moment = {m3c:>8.4f}")

print("\nIdentical mean and variance -- but the 3rd central moment is 0 for the")
print("symmetric distribution and positive for the lopsided one. Mean and variance")
print("alone cannot tell these apart; the third moment can.")

## 5. Visualizations

### The warm-up distribution and its powers

The same three probabilities weight different values as $k$ grows: odd powers keep the $-2$'s sign (pulling the moment negative), even powers turn it into the dominant positive contribution.

In [ ]:
values = np.array([-2, 0, 1])
pmf = np.array([1/3, 1/6, 1/2])

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharey=False)
for ax, k in zip(axes, range(1, 5)):
    contributions = pmf * values.astype(float)**k
    colors = ["tab:red" if c < 0 else "tab:blue" for c in contributions]
    ax.bar([str(v) for v in values], contributions, color=colors, edgecolor="black")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"k = {k}:  E[X^{k}] = {contributions.sum():.3f}")
    ax.set_xlabel("value x")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("contribution  p(x) · x^k")
plt.suptitle("Per-value contributions to the first four moments (blue positive, red negative)")
plt.tight_layout()
plt.show()

### Two distributions the first two moments cannot distinguish

Both shapes below have mean 0 and variance 0.5 — the bars tell the real story, and the third central moment is the first number that notices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

axes[0].bar([-1, 0, 1], [0.25, 0.5, 0.25], width=0.4, color="tab:blue",
            edgecolor="black")
axes[0].set_title("A: symmetric\nmean 0, var 0.5, 3rd central moment 0")

axes[1].bar([-0.354, 1.414], [0.8, 0.2], width=0.4, color="tab:orange",
            edgecolor="black")
axes[1].set_title("B: lopsided\nmean 0, var 0.5, 3rd central moment 0.53")

for ax in axes:
    ax.axvline(0, color="tab:red", linestyle="--", linewidth=1.5, label="mean = 0")
    ax.set_xlabel("value")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("probability")
plt.suptitle("Same first and second moments -- different distributions")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** A fair coin gives $X = 1$ for heads, $0$ for tails. Compute the first four moments. (Notice anything?)

**Exercise 2.** For the video's variable ($-2$ w.p. $\tfrac13$, $0$ w.p. $\tfrac16$, $1$ w.p. $\tfrac12$), compute the 5th moment by hand and check with the `moment` function.

### Intermediate

**Exercise 3.** Show that for *any* random variable, the first central moment $E[X - \mu]$ is exactly 0, and verify numerically for the die and the video's variable. (One line with linearity.)

**Exercise 4.** For a symmetric distribution (values $-a, 0, a$ with probabilities $p, 1-2p, p$), show numerically that **all odd central moments** vanish. Why does symmetry kill exactly the odd ones?

### Advanced

**Exercise 5.** For the standard normal (simulate or integrate), compute the first four moments. The known exact values are $0, 1, 0, 3$. The "3" for the fourth moment will become the reference point for kurtosis — every distribution is compared against it.

**Exercise 6.** Moments do not always exist! Simulate a Cauchy distribution (`rng.standard_cauchy`) and watch its running sample mean over 100,000 draws. Does it settle the way the die's mean did in the earlier notebooks? What does that suggest about the first moment of this distribution?

## 7. Solutions

In [ ]:
# Exercise 1: moments of a 0/1 coin
values = np.array([0, 1])
pmf = np.array([0.5, 0.5])
print("Exercise 1:", [moment(values, pmf, k) for k in range(1, 5)])
print("All moments equal 0.5: since 0^k = 0 and 1^k = 1, raising to any power")
print("changes nothing -- every moment is just P(X = 1).")

In [ ]:
# Exercise 2: the 5th moment of the video's variable
values = np.array([-2, 0, 1])
pmf = np.array([1/3, 1/6, 1/2])
# By hand: (1/3)(-32) + (1/6)(0) + (1/2)(1) = -32/3 + 1/2 = -61/6
print(f"Exercise 2: E[X^5] = (1/3)(-32) + (1/2)(1) = -61/6 = {-61/6:.4f}")
print(f"Function: {moment(values, pmf, 5):.4f}")

In [ ]:
# Exercise 3: first central moment is always zero
# E[X - mu] = E[X] - mu = 0 (linearity; mu is a constant).
faces = np.arange(1, 7)
die_pmf = np.full(6, 1 / 6)
video_values = np.array([-2, 0, 1])
video_pmf = np.array([1/3, 1/6, 1/2])
print(f"Exercise 3: die: {central_moment(faces, die_pmf, 1):.2e}, "
      f"video variable: {central_moment(video_values, video_pmf, 1):.2e}")

In [ ]:
# Exercise 4: odd central moments of a symmetric distribution
a, p = 3.0, 0.3
values = np.array([-a, 0.0, a])
pmf = np.array([p, 1 - 2 * p, p])

print("Exercise 4:")
for k in range(1, 7):
    print(f"  {k}-th central moment: {central_moment(values, pmf, k):>10.4f}")
print("Symmetry pairs every value x with a mirror value -x having the same")
print("probability. Odd powers give (x)^k = -(-x)^k, so each pair cancels;")
print("even powers give equal contributions that ADD instead.")

In [ ]:
# Exercise 5: first four moments of the standard normal
rng = np.random.default_rng(0)
z = rng.standard_normal(2_000_000)

print("Exercise 5:")
for k, exact in zip(range(1, 5), [0, 1, 0, 3]):
    print(f"  E[Z^{k}] simulated = {np.mean(z**k):>7.4f}   (exact {exact})")
print("The fourth moment of 3 is the normal benchmark that kurtosis compares against.")

In [ ]:
# Exercise 6: a distribution whose mean does not exist
rng = np.random.default_rng(0)

draws = rng.standard_cauchy(100_000)
running_mean = np.cumsum(draws) / np.arange(1, len(draws) + 1)

plt.figure(figsize=(9, 4))
plt.plot(running_mean, linewidth=0.8)
plt.title("Running sample mean of a Cauchy distribution: it never settles")
plt.xlabel("number of draws")
plt.ylabel("running mean")
plt.grid(True, alpha=0.3)
plt.show()

print("Exercise 6: unlike the die (which converged to 3.5), the Cauchy running")
print("mean keeps jumping -- occasional gigantic draws never stop dominating.")
print("Its tails are so heavy that E[X] (the first moment) does not exist.")

## 8. Conclusion

- The **$k$-th moment** of a random variable is $E[X^k] = \sum_i p_i x_i^k$ — the same weighted-average recipe with the values raised to the $k$-th power.
- The **first moment** is the mean; the **second moment** is related to the variance via the shortcut $\mathrm{Var}(X) = E[X^2] - E[X]^2$ (center first!).
- **Central moments** $E[(X-\mu)^k]$ are the descriptive ones: the 1st is always 0, the 2nd is the variance, the 3rd detects asymmetry, the 4th tail weight.
- Odd moments keep signs (they can cancel and detect lopsidedness); even moments amplify magnitudes (they measure spread and tails).
- Mean and variance cannot distinguish everything: distributions can agree on both and differ in higher moments — which is exactly why **skewness** (from the 3rd moment) and **kurtosis** (from the 4th) come next.